#Importação e carregamento dos dados

In [ ]:
import pandas as pd
import numpy as np

# Carregando a base de dados original via url raw do github
url_unodc = "https://raw.githubusercontent.com/Atila-dev/teste-topicos/refs/heads/main/data_cts_intentional_homicide.csv"

df_unodc = pd.read_csv(url_unodc)

### Limpeza e tratamento dos dados:


In [ ]:
# Criando uma cópia para preserwar o histórico
df_tratado = df_unodc.copy()

#Padronização da coluna de Idade
df_tratado['Age'] = df_tratado['Age'].astype(str).str.replace(' ', '')
df_tratado['Age'] = df_tratado['Age'].str.replace('60andolder', '60+')

#Tratando as vírgulas decimais da coluna VALUE
df_tratado['VALUE'] = pd.to_numeric(df_tratado['VALUE'].astype(str).str.replace(',', '.'), errors='coerce')

#Separação de Dados

Como a ONU já fez uma pré-análise dos dados, dividimos a tabela em duas separadas para que possamos fazer a nossa própria análise e, se necessário, utilizar a da ONU.

In [ ]:
# Dataset exclusivo de Países
df_paises = df_tratado[
    (~df_tratado['Indicator'].str.contains('Regional Estimate', na=False)) &
    (~df_tratado['Iso3_code'].str.startswith(('M49', 'BIG5', 'WORLD'), na=False))
].copy()

# Exportando para planilha de países
nome_paises = "dataset_homicidios_paises_tratado.csv"
df_paises.to_csv(nome_paises, index=False)

# Dataset exclusivo de Estimativas Regionais da ONU
df_onu = df_tratado[
    (df_tratado['Indicator'].str.contains('Regional Estimate', na=False)) |
    (df_tratado['Iso3_code'].str.startswith(('M49', 'BIG5', 'WORLD'), na=False))
].copy()

# Exportando para planilha de regiões
nome_onu = "dataset_homicidios_regioes_onu_tratado.csv"
df_onu.to_csv(nome_onu, index=False)

#Visões dos dados

In [ ]:
display(df_paises.shape)
display(df_onu.shape)

(110862, 13)

(2772, 13)

In [ ]:
display(df_paises.describe())
print("\n")
display(df_onu.describe())

,Year,VALUE
count,110862.000000,110862.000000
mean,2014.789116,233.337625
std,6.102654,2031.347567
min,1990.000000,0.000000
25%,2012.000000,0.233699
50%,2016.000000,2.353695
75%,2019.000000,19.335443
max,2022.000000,63788.000000


,Year,VALUE
count,2772.000000,2772.000000
mean,2010.500000,20450.234640
std,6.345433,58191.254718
min,2000.000000,0.458624
25%,2005.000000,3.549843
50%,2010.500000,23.163526
75%,2016.000000,10813.644655
max,2021.000000,457945.485000


In [ ]:
print(f"Valores Nulos: {df_paises.isnull().sum().sum()}")
print(f"Linhas Duplicadas: {df_paises.duplicated().sum()}")
print(f"Valores Nulos (ONU): {df_onu.isnull().sum().sum()}")
print(f"Linhas Duplicadas (ONU): {df_onu.duplicated().sum()}")

Valores Nulos: 0
Linhas Duplicadas: 0
Valores Nulos (ONU): 0
Linhas Duplicadas (ONU): 0


# Análise Exploratória

##1 - Quais países apresentaram os 10 maiores índices de homicídios nos últimos 5 anos?


* Utilizou-se o DataFrame `df_paises`, pois a requisição especifica entidades nacionais, excluindo os dados macro da ONU.

* A seleção de `Rate per 100,000 population` foi aplicada. O uso de números absolutos (`Counts`) resultaria em um viés populacional, onde países com maior número de habitantes apareceriam no topo do ranking independentemente da severidade real do índice de criminalidade.

* O método `.mean()` foi escolhido para calcular a média do índice ao longo dos 5 anos, estabelecendo um valor representativo do período solicitado em vez de selecionar um pico isolado de um único ano.

In [ ]:
# Definição do período de análise
anos_alvo = [2018, 2019, 2020, 2021, 2022]

# Filtragem
df_taxa = df_paises[
    (df_paises['Year'].isin(anos_alvo)) &
    (df_paises['Indicator'] == 'Victims of intentional homicide') &
    (df_paises['Unit of measurement'] == 'Rate per 100,000 population') &
    (df_paises['Sex'] == 'Total') &
    (df_paises['Age'] == 'Total') &
    (df_paises['Dimension'] == 'Total') &
    (df_paises['Category'] == 'Total')
].copy()

# Agrupamento por país e cálculo da média da taxa no período de 5 anos
taxa_media_paises = df_taxa.groupby('Country')['VALUE'].mean().reset_index()

# Ordenação descendente e seleção dos 10 primeiros
top_10_paises = taxa_media_paises.sort_values(by='VALUE', ascending=False).head(10)

# Renomeação da coluna para adequação de nomenclatura
top_10_paises = top_10_paises.rename(columns={'VALUE': 'Taxa Média (por 100 mil hab.)'})

# Exibição da tabela final
display(top_10_paises.reset_index(drop=True))

,Country,Taxa Média (por 100 mil hab.)
0,Jamaica,49.237990
1,Honduras,37.571005
2,South Africa,35.779754
3,Trinidad and Tobago,32.996442
4,Belize,31.365003
5,Saint Kitts and Nevis,30.926257
6,Saint Vincent and the Grenadines,30.603680
7,Saint Lucia,30.245971
8,Venezuela (Bolivarian Republic of),29.905767
9,Mexico,28.471548


##2 - Quais países apresentam os 10 maiores índices de homicídios de mulheres em 2022?

* O uso da taxa por 100.000 habitantes foi mantido. Utilizar contagens absolutas (`Counts`) em recortes de gênero introduz distorções severas, pois países com grandes populações femininas liderariam a lista em números absolutos, ocultando a real proporção do risco letal no país.

In [ ]:
# Filtragem
df_mulheres_2022 = df_paises[
    (df_paises['Year'] == 2022) &
    (df_paises['Sex'] == 'Female') &
    (df_paises['Indicator'] == 'Victims of intentional homicide') &
    (df_paises['Unit of measurement'] == 'Rate per 100,000 population') &
    (df_paises['Age'] == 'Total') &
    (df_paises['Dimension'] == 'Total') &
    (df_paises['Category'] == 'Total')
].copy()

# Ordenação
top_10_mulheres_2022 = df_mulheres_2022.sort_values(by='VALUE', ascending=False).head(10)

# Isolamento das colunas de interesse e renomeação para apresentação
top_10_mulheres_2022 = top_10_mulheres_2022[['Country', 'VALUE']]
top_10_mulheres_2022 = top_10_mulheres_2022.rename(columns={'VALUE': 'Taxa de Homicídios (Mulheres por 100 mil hab.)'})

# Exibição da tabela resultante
display(top_10_mulheres_2022.reset_index(drop=True))

,Country,Taxa de Homicídios (Mulheres por 100 mil hab.)
0,Saint Lucia,9.900010
1,Saint Vincent and the Grenadines,9.800174
2,Jamaica,8.701764
3,Mexico,6.014469
4,Bahamas,4.669264
5,Ecuador,4.647089
6,Belize,4.465936
7,Latvia,4.031317
8,Colombia,3.810115
9,Suriname,3.221489


* Ao analisar dados de homicídios filtrados por gênero, avaliar apenas a taxa isolada pode omitir o contexto da dinâmica criminal do país. Um tratamento analítico adicional de alto valor é calcular a **proporção de vítimas femininas em relação ao total de vítimas** nesses mesmos 10 países. Isso permite identificar se o alto índice reflete uma criminalidade geral elevada no país ou um problema de letalidade desproporcionalmente focado em mulheres.

* Para realizar esta análise de proporção, é mandatório utilizar a unidade `Counts` (números absolutos), pois taxas não podem ser somadas ou divididas diretamente com precisão demográfica.

In [ ]:
# Obtenção da lista dos 10 países identificados anteriormente
paises_alvo = top_10_mulheres_2022['Country'].tolist()

# Filtro base para valores absolutos (Counts) em 2022 para os países do Top 10
df_counts_2022 = df_paises[
    (df_paises['Country'].isin(paises_alvo)) &
    (df_paises['Year'] == 2022) &
    (df_paises['Indicator'] == 'Victims of intentional homicide') &
    (df_paises['Unit of measurement'] == 'Counts') &
    (df_paises['Age'] == 'Total') &
    (df_paises['Dimension'] == 'Total') &
    (df_paises['Category'] == 'Total')
]

# Extração dos totais femininos
df_fem_counts = df_counts_2022[df_counts_2022['Sex'] == 'Female'][['Country', 'VALUE']]
df_fem_counts = df_fem_counts.rename(columns={'VALUE': 'Vítimas Mulheres'})

# Extração dos totais gerais
df_total_counts = df_counts_2022[df_counts_2022['Sex'] == 'Total'][['Country', 'VALUE']]
df_total_counts = df_total_counts.rename(columns={'VALUE': 'Total de Vítimas'})

# Mesclagem dos dados e cálculo percentual
df_proporcao = pd.merge(df_fem_counts, df_total_counts, on='Country')
df_proporcao['Proporção Feminina (%)'] = (df_proporcao['Vítimas Mulheres'] / df_proporcao['Total de Vítimas']) * 100

# Ordenação pela proporção e exibição
df_proporcao = df_proporcao.sort_values(by='Proporção Feminina (%)', ascending=False)
display(df_proporcao.reset_index(drop=True))

,Country,Vítimas Mulheres,Total de Vítimas,Proporção Feminina (%)
0,Latvia,40.0,67.0,59.701493
1,Suriname,10.0,45.0,22.222222
2,Saint Lucia,9.0,66.0,13.636364
3,Saint Vincent and the Grenadines,5.0,42.0,11.904762
4,Mexico,3928.0,33287.0,11.800403
5,Ecuador,419.0,4859.0,8.623173
6,Jamaica,124.0,1508.0,8.222812
7,Belize,9.0,113.0,7.964602
8,Bahamas,10.0,128.0,7.812500
9,Colombia,1002.0,13166.0,7.610512


##3 - Quais as regiões com mais homicídios?

* A escolha por `Counts` (valores absolutos) foi feita porque a pergunta busca saber onde há "mais homicídios" fisicamente, o que requer a contagem direta de vítimas. A taxa por 100 mil habitantes indicaria o risco letal proporcional, não o volume total.

* Utilizou-se a função `.groupby('Region')` no DataFrame `df_paises`. Isso permite somar os dados de todos os países que compõem cada continente/região de forma controlada.

In [ ]:
# Filtragem
df_regioes = df_paises[
    (df_paises['Indicator'] == 'Victims of intentional homicide') &
    (df_paises['Unit of measurement'] == 'Counts') &
    (df_paises['Sex'] == 'Total') &
    (df_paises['Age'] == 'Total') &
    (df_paises['Dimension'] == 'Total') &
    (df_paises['Category'] == 'Total')
].copy()

# Agrupamento pela coluna 'Region' e soma de todos os registros históricos
ranking_regioes = df_regioes.groupby('Region')['VALUE'].sum().reset_index()

# Ordenação
ranking_regioes = ranking_regioes.sort_values(by='VALUE', ascending=False)

# Renomeação da coluna para clareza dos dados apresentados
ranking_regioes = ranking_regioes.rename(columns={'VALUE': 'Total de Homicídios'})

# Exibição da tabela
display(ranking_regioes.reset_index(drop=True))

,Region,Total de Homicídios
0,Americas,4.357154e+06
1,Asia,2.852197e+06
2,Europe,1.222767e+06
3,Africa,9.548166e+05
4,Oceania,1.840745e+04


##4 - Quais países com menor número de homicídios em cada sub-região?

* O método `.groupby(['Subregion', 'Country'])` unifica os dados de todos os anos para cada país.

* A função `.idxmin()` foi selecionada por ser a forma mais eficiente no Pandas para retornar o índice da linha que contém o valor mínimo dentro de um grupo específico. Após obter os índices, a função `.loc[]` resgata as linhas completas, mantendo o nome do país associado ao valor mínimo.

In [ ]:
# Filtragem
df_subregiao = df_paises[
    (df_paises['Indicator'] == 'Victims of intentional homicide') &
    (df_paises['Unit of measurement'] == 'Counts') &
    (df_paises['Sex'] == 'Total') &
    (df_paises['Age'] == 'Total') &
    (df_paises['Dimension'] == 'Total') &
    (df_paises['Category'] == 'Total')
].copy()

# Agrupamento por Sub-região e País para somar o total histórico de homicídios
total_pais_subregiao = df_subregiao.groupby(['Subregion', 'Country'])['VALUE'].sum().reset_index()

# Obtenção dos índices com o valor mínimo de homicídios em cada sub-região
indices_menores = total_pais_subregiao.groupby('Subregion')['VALUE'].idxmin()

# Filtragem do dataframe original utilizando os índices encontrados
paises_menores_homicidios = total_pais_subregiao.loc[indices_menores]

# Ordenação por sub-região e renomeação da coluna de valores
paises_menores_homicidios = paises_menores_homicidios.sort_values(by=['Subregion'])
paises_menores_homicidios = paises_menores_homicidios.rename(columns={'VALUE': 'Total de Homicídios'})

# Exibição da tabela resultante
display(paises_menores_homicidios.reset_index(drop=True))

,Subregion,Country,Total de Homicídios
0,Australia and New Zealand,New Zealand,1395.817680
1,Central Asia,Turkmenistan,5328.000000
2,Eastern Asia,"China, Macao Special Administrative Region",244.000000
3,Eastern Europe,Slovakia,3091.000000
4,Latin America and the Caribbean,Montserrat,9.000000
5,Melanesia,Vanuatu,3.000000
6,Micronesia,Micronesia (Federated States of),1.000000
7,Northern Africa,Tunisia,3263.000000
8,Northern America,Saint Pierre and Miquelon,2.000000
9,Northern Europe,Isle of Man,9.000000


* Avaliar a soma total de anos não especificados gera um problema estatístico: por exemplo, um país que forneceu dados para a ONU durante apenas 1 ano poderá registrar um total acumulado menor do que um microestado seguro que forneceu dados consistentemente por 20 anos.

* Para realizar uma comparação estatística justa entre os países de cada sub-região, a métrica apropriada é a **Média Anual de Homicídios**, em vez da soma bruta do período.

In [ ]:
# Agrupamento para encontrar a média anual de homicídios por país
media_anual_pais = df_subregiao.groupby(['Subregion', 'Country'])['VALUE'].mean().reset_index()

# Obtenção dos índices com a menor média anual dentro de cada sub-região
indices_menores_medias = media_anual_pais.groupby('Subregion')['VALUE'].idxmin()

# Filtragem das linhas correspondentes
menores_medias_subregiao = media_anual_pais.loc[indices_menores_medias]

# Ordenação e formatação final
menores_medias_subregiao = menores_medias_subregiao.sort_values(by='Subregion')
menores_medias_subregiao = menores_medias_subregiao.rename(columns={'VALUE': 'Média Anual de Homicídios'})

# Exibição do resultado padronizado
display(menores_medias_subregiao.reset_index(drop=True))

,Subregion,Country,Média Anual de Homicídios
0,Australia and New Zealand,New Zealand,49.850631
1,Central Asia,Turkmenistan,231.652174
2,Eastern Asia,"China, Macao Special Administrative Region",7.625000
3,Eastern Europe,Slovakia,96.593750
4,Latin America and the Caribbean,Montserrat,0.428571
5,Melanesia,Vanuatu,1.500000
6,Micronesia,Micronesia (Federated States of),1.000000
7,Northern Africa,Tunisia,326.300000
8,Northern America,Saint Pierre and Miquelon,0.500000
9,Northern Europe,Isle of Man,1.500000


##5 - Quais países com menor número de mortes de mulheres?

* O filtro `Sex == 'Female'` restringe a busca ao sexo feminino. As demais variáveis (`Age, Dimension, Category`) foram fixadas em `Total` para garantir a integridade da soma, impedindo a contagem duplicada de subcategorias metodológicas.

In [ ]:
# Filtragem
df_menor_mulheres = df_paises[
    (df_paises['Sex'] == 'Female') &
    (df_paises['Indicator'] == 'Victims of intentional homicide') &
    (df_paises['Unit of measurement'] == 'Counts') &
    (df_paises['Age'] == 'Total') &
    (df_paises['Dimension'] == 'Total') &
    (df_paises['Category'] == 'Total')
].copy()

# Agrupamento por país e soma de todos os registros históricos disponíveis
total_mulheres_pais = df_menor_mulheres.groupby('Country')['VALUE'].sum().reset_index()

# Ordenação
top_10_menores_mulheres = total_mulheres_pais.sort_values(by='VALUE', ascending=True)

# Renomeação
top_10_menores_mulheres = top_10_menores_mulheres.rename(columns={'VALUE': 'Total de Vítimas Mulheres'})

# Exibição da tabela
display(top_10_menores_mulheres.reset_index(drop=True))

,Country,Total de Vítimas Mulheres
0,Holy See,0.000000
1,Sao Tome and Principe,0.000000
2,Saint Pierre and Miquelon,0.000000
3,Monaco,0.000000
4,Micronesia (Federated States of),1.000000
...,...,...
155,Mexico,69527.626782
156,Brazil,122059.786112
157,United States of America,133480.479492
158,Russian Federation,214851.000000


* A utilização de números absolutos na busca por menores índices criminais gera um viés demográfico severo. Microestados invariavelmente exibirão os menores números absolutos devido à sua população reduzida, enquanto nações que falharam em reportar dados para a ONU na maioria dos anos apresentarão somas acumuladas artificialmente baixas.

* O tratamento adequado para estabelecer uma comparação proporcional e justa sobre a segurança da população feminina entre os países exige o uso da taxa populacional e a extração da média anual.

In [ ]:
# Filtragem utilizando a taxa proporcional (Rate) em vez de valores absolutos
df_taxa_mulheres = df_paises[
    (df_paises['Sex'] == 'Female') &
    (df_paises['Indicator'] == 'Victims of intentional homicide') &
    (df_paises['Unit of measurement'] == 'Rate per 100,000 population') &
    (df_paises['Age'] == 'Total') &
    (df_paises['Dimension'] == 'Total') &
    (df_paises['Category'] == 'Total')
].copy()

# Cálculo da média histórica da taxa para nivelar anos não reportados
media_taxa_mulheres = df_taxa_mulheres.groupby('Country')['VALUE'].mean().reset_index()

# Ordenação
top_10_menores_taxas_mulheres = media_taxa_mulheres.sort_values(by='VALUE', ascending=True)

# Renomeação
top_10_menores_taxas_mulheres = top_10_menores_taxas_mulheres.rename(columns={'VALUE': 'Taxa Média de Homicídios (Mulheres por 100 mil hab.)'})

# Exibição da tabela tratada
display(top_10_menores_taxas_mulheres.reset_index(drop=True))

,Country,Taxa Média de Homicídios (Mulheres por 100 mil hab.)
0,Saint Pierre and Miquelon,0.000000
1,Sao Tome and Principe,0.000000
2,Monaco,0.000000
3,Bahrain,0.132847
4,Singapore,0.215083
...,...,...
154,Russian Federation,8.602723
155,Honduras,9.014499
156,South Africa,9.691162
157,Jamaica,9.857642


##6 - Quais as sub-regiões com maior número de homicídios?

* Para evitar contagens duplicadas, são considerados apenas registros
classificados como Total para sexo, faixa etária, dimensão e categoria.
Dessa forma cada homicídio é contabilizado apenas uma vez.

* A análise é realizada utilizando o número absoluto de vítimas
 (Counts), permitindo identificar quais sub-regiões concentram
 o maior volume total de homicídios registrados.

* Também é calculada a taxa média por 100 mil habitantes.
Essa métrica reduz o efeito do tamanho populacional e permite
comparações mais justas entre sub-regiões.

In [ ]:
df_count = df_paises[
    (df_paises['Indicator'] == 'Victims of intentional homicide') &
    (df_paises['Unit of measurement'] == 'Counts') &
    (df_paises['Sex'] == 'Total') &
    (df_paises['Age'] == 'Total') &
    (df_paises['Dimension'] == 'Total') &
    (df_paises['Category'] == 'Total')
]

ranking_count = (
    df_count.groupby('Subregion')['VALUE']
    .sum()
    .sort_values(ascending=False)
)

print('Top 10 sub-regiões por volume de homicídios')
display(ranking_count.head(10))

df_rate = df_paises[
    (df_paises['Indicator'] == 'Victims of intentional homicide') &
    (df_paises['Unit of measurement'] == 'Rate per 100,000 population') &
    (df_paises['Sex'] == 'Total') &
    (df_paises['Age'] == 'Total') &
    (df_paises['Dimension'] == 'Total') &
    (df_paises['Category'] == 'Total')
]

ranking_rate = (
    df_rate.groupby('Subregion')['VALUE']
    .mean()
    .sort_values(ascending=False)
)

print('Top 10 sub-regiões por taxa média')
display(ranking_rate.head(10))


Top 10 sub-regiões por volume de homicídios


,VALUE
Subregion,
Latin America and the Caribbean,3.784868e+06
Southern Asia,1.883914e+06
Eastern Europe,1.034177e+06
Sub-Saharan Africa,9.093686e+05
Northern America,5.722860e+05
Eastern Asia,3.798067e+05
South-eastern Asia,3.427134e+05
Western Asia,1.647560e+05
Central Asia,8.100700e+04


Top 10 sub-regiões por taxa média


,VALUE
Subregion,
Latin America and the Caribbean,19.414044
Sub-Saharan Africa,9.950646
Northern America,6.867587
Central Asia,6.279544
Eastern Europe,5.003637
Polynesia,4.698450
Melanesia,4.511491
Southern Asia,3.863405
South-eastern Asia,3.442672


##7 - País com maior número de homicídios em cada continente em 2020

* A análise utiliza números absolutos de vítimas (Counts),
 pois o objetivo é identificar o país com maior volume de homicídios
 em cada continente.

* Após o agrupamento por continente e país, seleciona-se o maior valor
 de cada continente para identificar seu respectivo líder em homicídios

In [ ]:
df_2020 = df_paises[
    (df_paises['Year'] == 2020) &
    (df_paises['Indicator'] == 'Victims of intentional homicide') &
    (df_paises['Unit of measurement'] == 'Counts') &
    (df_paises['Sex'] == 'Total') &
    (df_paises['Age'] == 'Total') &
    (df_paises['Dimension'] == 'Total') &
    (df_paises['Category'] == 'Total')
]

homicidios = (
    df_2020.groupby(['Region', 'Country'])['VALUE']
    .sum()
    .reset_index()
)

idx = homicidios.groupby('Region')['VALUE'].idxmax()

resultado = homicidios.loc[idx].sort_values('VALUE', ascending=False)

display(resultado)

,Region,Country,VALUE
25,Americas,Brazil,47722.0
64,Asia,India,40651.0
13,Africa,South Africa,19972.0
117,Europe,Russian Federation,10697.0
128,Oceania,Australia,221.0


##8 - Qual o país mais violento para mulheres em 2021?

* Nesta análise foi utilizada a taxa por 100 mil habitantes.
* Será mostrado que utilizar o volume absoluto favorece e enviesa países muito populosos.

* Foram considerados apenas registros referentes ao sexo feminino,
 mantendo os demais atributos em Total para evitar duplicidade.

* O resultado representa o país com maior incidência proporcional
 de homicídios de mulheres no ano de 2021.

In [ ]:
df_mulheres_count = df_paises[
    (df_paises['Year'] == 2021) &
    (df_paises['Sex'] == 'Female') &
    (df_paises['Indicator'] == 'Victims of intentional homicide') &
    (df_paises['Unit of measurement'] == 'Counts') &
    (df_paises['Age'] == 'Total') &
    (df_paises['Dimension'] == 'Total') &
    (df_paises['Category'] == 'Total')
]

ranking_mulheres_count = (
    df_mulheres_count.groupby('Country')['VALUE']
    .mean()
    .sort_values(ascending=False)
)

print('Países mais violentos para mulheres por Volume')
display(ranking_mulheres_count.head(10))

df_mulheres_rate = df_paises[
    (df_paises['Year'] == 2021) &
    (df_paises['Sex'] == 'Female') &
    (df_paises['Indicator'] == 'Victims of intentional homicide') &
    (df_paises['Unit of measurement'] == 'Rate per 100,000 population') &
    (df_paises['Age'] == 'Total') &
    (df_paises['Dimension'] == 'Total') &
    (df_paises['Category'] == 'Total')
]

ranking_mulheres_rate = (
    df_mulheres_rate.groupby('Country')['VALUE']
    .mean()
    .sort_values(ascending=False)
)

print('Países mais violentos para mulheres por índice')
display(ranking_mulheres_rate.head(10))



Países mais violentos para mulheres por Volume


,VALUE
Country,
India,17012.375770
United States of America,4973.774454
Mexico,4002.000000
Brazil,3844.000000
Russian Federation,2568.000000
Myanmar,1969.000000
Colombia,957.000000
Kenya,706.000000
Guatemala,545.000000


Países mais violentos para mulheres por índice


,VALUE
Country,
Antigua and Barbuda,10.265149
Jamaica,9.335017
Saint Lucia,8.819168
Botswana,7.629581
Myanmar,7.288432
Namibia,6.799161
Honduras,6.466649
Mexico,6.171259
Guatemala,6.129440


##9 - Qual país possui a maior média de vítimas de homicídio intencional?

* Em vez de utilizar a soma total de homicídios ao longo dos anos,
foi calculada a média por país.

* Essa abordagem reduz o impacto de países que possuem mais anos
registrados na base de dados.
* O objetivo é identificar quais países apresentam, em média,
os maiores volumes anuais de homicídios intencionais.

* Assim como nas análises anteriores, foram mantidos apenas registros
classificados como Total para evitar múltiplas contagens do mesmo evento.

In [ ]:
df_vitimas = df_paises[
    (df_paises['Indicator'] == 'Victims of intentional homicide') &
    (df_paises['Unit of measurement'] == 'Counts') &
    (df_paises['Sex'] == 'Total') &
    (df_paises['Age'] == 'Total') &
    (df_paises['Dimension'] == 'Total') &
    (df_paises['Category'] == 'Total')
]

ranking_media = (
    df_vitimas.groupby('Country')['VALUE']
    .mean()
    .sort_values(ascending=False)
)

display(ranking_media.head(10))


,VALUE
Country,
Nigeria,53800.000000
Brazil,46316.679879
India,45624.062500
Russian Federation,27121.906250
South Africa,20280.703704
Mexico,19386.636364
Colombia,19383.272727
United States of America,18377.200000
China,15630.727273


##10 - Qual a média anual de homicídios no Brasil nos últimos 10 anos?

* A análise foi limitada ao Brasil e aos anos disponíveis no conjunto de dados.

* Embora o filtro permita anos até 2023, a base utilizada contém registros
 para o Brasil apenas até 2021, portanto o periodo  de 10 anos vai de 2012 até 2021.
* Inicialmente os dados são agrupados por ano para obter o total anual
 de homicídios registrados no país.
* A média final é calculada a partir dos totais anuais.
 Dessa forma obtém-se a média de homicídios por ano.
* Também é calculada a taxa média por 100 mil habitantes no período,
 permitindo uma interpretação complementar ao número absoluto de vítimas.

In [ ]:
df_brasil = df_paises[
    (df_paises['Country'] == 'Brazil') &
    (df_paises['Year'].between(2012, 2021)) &
    (df_paises['Indicator'] == 'Victims of intentional homicide') &
    (df_paises['Unit of measurement'] == 'Counts') &
    (df_paises['Sex'] == 'Total') &
    (df_paises['Age'] == 'Total') &
    (df_paises['Dimension'] == 'Total') &
    (df_paises['Category'] == 'Total')
]

media_por_ano = (
    df_brasil.groupby('Year')['VALUE']
    .sum()
)

display(media_por_ano)

media_homicidios = media_por_ano.mean()

print(f'Média anual de homicídios no Brasil (2012-2021): {media_homicidios:.2f}')

df_brasil_taxa = df_paises[
    (df_paises['Country'] == 'Brazil') &
    (df_paises['Year'].between(2012, 2021)) &
    (df_paises['Indicator'] == 'Victims of intentional homicide') &
    (df_paises['Unit of measurement'] == 'Rate per 100,000 population') &
    (df_paises['Sex'] == 'Total') &
    (df_paises['Age'] == 'Total') &
    (df_paises['Dimension'] == 'Total') &
    (df_paises['Category'] == 'Total')
]

print(f"Taxa média no período: {df_brasil_taxa['VALUE'].mean():.2f}")

,VALUE
Year,
2012,56388.0
2013,56845.0
2014,59733.0
2015,58184.0
2016,61208.0
2017,63788.0
2018,55980.0
2019,44073.0
2020,47722.0


Média anual de homicídios no Brasil (2012-2021): 54948.30
Taxa média no período: 26.54
